# Write the dust devil likelihood GeoTIFF

Run after `dust_devil_likelihood_demo.ipynb`, in its own kernel. Needs numpy and rasterio only (no matplotlib, no pandas).
Copies the CRS and transform from the THEMIS template, so the output lines up with the other layers. Band 1 = daily-mean score, band 2 = max over the day.

In [1]:
from pathlib import Path
import json
import numpy as np
import rasterio

PROC = Path("processed_data")
DEMO_SOL = 800
TEMPLATE_TIF = PROC / "gale_ti_night_themis100m_native.tif"
META_JSON = PROC / "gale_dust_devil_meta.json"
NPZ = PROC / f"gale_dust_devil_likelihood_sol{DEMO_SOL}.npz"
OUT = PROC / f"gale_dust_devil_likelihood_sol{DEMO_SOL}.tif"

meta = json.loads(META_JSON.read_text())
z = np.load(NPZ)
day_ltst = z["day_ltst"]
with rasterio.open(TEMPLATE_TIF) as t:
    crs, tf, shape = t.crs, t.transform, (t.height, t.width)
assert shape == tuple(meta["template_shape"]) == z["likelihood_mean"].shape, "template, meta.json and npz come from different grids"
assert np.allclose(list(tf)[:6], meta["template_transform"]), "meta.json transform differs from the template tif"
assert crs.is_geographic, "template should be lon/lat"

profile = dict(driver="GTiff", dtype="float32", count=2, height=shape[0], width=shape[1], crs=crs, transform=tf, nodata=np.nan)
with rasterio.open(OUT, "w", **profile) as dst:
    dst.write(z["likelihood_mean"], 1); dst.set_band_description(1, f"likelihood_mean_{day_ltst[0]:g}-{day_ltst[1]:g}LTST")
    dst.write(z["likelihood_max"], 2);  dst.set_band_description(2, "likelihood_max_over_day")
print("wrote", OUT)

wrote processed_data\gale_dust_devil_likelihood_sol800.tif
